# Getis-Ord Gi*: find local concentrations

This lab identifies local concentrations of high and low values. We will build a binary neighbour graph, include the focal unit in each local sum, calculate Gi* z-scores and permutation p-values, then apply an FDR cutoff.

**You need:** Python, NumPy, Matplotlib, libpysal and esda.

The ring is a synthetic graph, not a geographic map. The values are positive so that the concentration question is meaningful.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from libpysal.weights import W
from esda import G_Local, fdr


## 1. Build a binary neighbour graph

Each unit is connected to the two units beside it. We leave the diagonal out of the weights matrix and ask Gi* to include the focal observation with `star=True`.

In [ ]:
n = 12
neighbors = {i: [(i - 1) % n, (i + 1) % n] for i in range(n)}
w = W(neighbors)
values = np.array([10, 11, 9, 2, 1, 2, 8, 9, 10, 1, 2, 1], dtype=float)

gi_star = G_Local(
    values,
    w,
    transform='B',
    star=True,
    permutations=999,
    seed=12345,
    alternative='two-sided',
)

print('Observed values:', values.tolist())
print('Gi* z-scores:', np.round(gi_star.Zs, 3))
print('Permutation p-values:', np.round(gi_star.p_sim, 3))


In [ ]:
angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
xy = np.column_stack([np.cos(angles), np.sin(angles)])
fig, ax = plt.subplots(figsize=(6, 6))
for i, linked in neighbors.items():
    for j in linked:
        if i < j:
            ax.plot([xy[i, 0], xy[j, 0]], [xy[i, 1], xy[j, 1]], linewidth=2)
ax.scatter(xy[:, 0], xy[:, 1], s=850, zorder=3)
for i, (x, y) in enumerate(xy):
    ax.text(x, y, f'U{i}\\n{values[i]:g}', ha='center', va='center', color='white', weight='bold')
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Synthetic ring graph: unit ID and positive value')
plt.show()


## 2. Read the local z-scores

A positive z-score points toward a local concentration of high values. A negative z-score points toward a concentration of low values. The p-value helps assess whether the result is unusual under the permutation null.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(np.arange(n), gi_star.Zs)
ax.axhline(0, linewidth=1)
ax.set_xticks(np.arange(n))
ax.set_xticklabels([f'U{i}' for i in range(n)])
ax.set_xlabel('Spatial unit')
ax.set_ylabel('Gi* z-score')
ax.set_title('Local concentration by unit')
plt.show()


## 3. Apply false-discovery-rate control

We run one local test per unit. The FDR function returns a p-value cutoff. We use that cutoff together with the sign of the z-score to label hotspots and coldspots.

In [ ]:
fdr_cutoff = fdr(gi_star.p_sim, alpha=0.05)
passes_fdr = gi_star.p_sim <= fdr_cutoff
hotspots = passes_fdr & (gi_star.Zs > 0)
coldspots = passes_fdr & (gi_star.Zs < 0)

print('FDR cutoff:', fdr_cutoff)
print('Hotspots:', np.flatnonzero(hotspots).tolist())
print('Coldspots:', np.flatnonzero(coldspots).tolist())
print('Unit | value | Gi* z-score | p-value | classification')
for i in range(n):
    label = 'hotspot' if hotspots[i] else 'coldspot' if coldspots[i] else 'not significant'
    print(f'U{i:02d} | {values[i]:g} | {gi_star.Zs[i]: .3f} | {gi_star.p_sim[i]:.3f} | {label}')


## 4. Change the values, keep the graph

Try a more even distribution of positive values or create a single very high value among moderate values. Keep the same graph and rerun the Gi* calculation. Notice that a hotspot is about the concentration across the focal unit's neighbourhood, not just the focal value alone.

In [ ]:
changed = np.array([3, 3, 3, 3, 20, 3, 3, 3, 3, 3, 3, 3], dtype=float)
gi_changed = G_Local(
    changed,
    w,
    transform='B',
    star=True,
    permutations=999,
    seed=12345,
    alternative='two-sided',
)
print('Changed values:', changed.tolist())
print('Gi* z-scores:', np.round(gi_changed.Zs, 3))
print('Permutation p-values:', np.round(gi_changed.p_sim, 3))


## What to take away

- Gi* measures concentration in a neighbourhood, including the focal unit.
- Positive z-scores indicate high-value concentrations; negative z-scores indicate low-value concentrations.
- Binary weights and focal-unit inclusion are important parts of the specification.
- A hotspot is not the same as a high-valued spatial outlier.
- Local tests require a multiple-testing strategy and a clearly reported inference method.

Continue with the [Getis-Ord Gi* lesson](../algorithms/statistics/getis-ord-gi-star.md), [Local Moran's I](../algorithms/statistics/local-morans-i.md) and the [global spatial autocorrelation path](../learning-paths/global-spatial-autocorrelation.md).

**Reference:** [PySAL's Getis-Ord hotspot guide](https://pysal.org/esda/stable/user-guide/local_getisord.html) and [G_Local API reference](https://pysal.org/esda/stable/generated/esda.G_Local.html).